# Mission 1 - 신고자 성별 분류

## 1. 실험 개요

본 실험의 목표는 119 신고 통화 음성으로부터 신고자의 성별(M/F)을 분류하는 것입니다.

- 학습에는 서울 Training 데이터만 사용했습니다.
- Validation 데이터는 학습에 사용하지 않고 평가에만 사용했습니다.
- annotation에서는 startAt, endAt, Speaker 정보를 신고자 발화 구간 전처리에만 사용했습니다.
- gender는 학습 target과 Validation 평가 정답으로만 사용했으며, text·symptom·address 등 다른 annotation 정보는 입력으로 사용하지 않았습니다.

실험 흐름은 Log-Mel CNN → MFCC+F0 RandomForest → WavLM 1-chunk → WavLM 3-chunk → late fusion 비교 순서입니다. 최종 제출 모델은 Validation에서 안정적으로 높은 성능을 보인 WavLM-base-plus 3-chunk inference 단일 모델입니다.

## 2. 경로 및 실험 환경 설정

아래 셀은 학습 데이터, 코드, 모델 저장 경로를 설정합니다. 이후의 학습 코드와 실험 로그는 당시 사용한 코드를 그대로 보존합니다.

In [ ]:
from pathlib import Path

DATA_ROOT = "/content/drive/MyDrive/data(inho)"
CODE_DIR = "/content/drive/MyDrive/mission1"
SAVE_DIR = "/content/drive/MyDrive/saved_models/mission1"

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

print("데이터:", Path(DATA_ROOT).exists())
print("코드:", Path(CODE_DIR).exists())
print("저장폴더:", Path(SAVE_DIR).exists())

print()
print("common.py:", Path(CODE_DIR, "common.py").exists())
print("train_cnn.py:", Path(CODE_DIR, "train_cnn.py").exists())
print("train_mfcc_rf.py:", Path(CODE_DIR, "train_mfcc_rf.py").exists())

데이터: True
코드: True
저장폴더: True

common.py: True
train_cnn.py: True
train_mfcc_rf.py: True


## 3. 데이터셋 확인 및 전처리

12개 part의 서울 Training·Validation 데이터를 확인했습니다. WAV와 JSON은 파일 stem 기준으로 매칭했으며, Train 29,200개·Validation 3,640개 모두 누락 없이 대응합니다.

신고자 음성은 annotation의 startAt, endAt, Speaker 정보를 이용해 speaker=1 발화 구간만 이어 붙여 사용합니다.

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/data(inho)")

parts = sorted([
    p for p in ROOT.iterdir()
    if p.is_dir()
])

print("part 개수:", len(parts))

for p in parts:
    print(p.name)

part 개수: 12
대학부 데이터-20260827T163800Z-1-001
대학부 데이터-20260827T163800Z-1-002
대학부 데이터-20260827T163800Z-1-003
대학부 데이터-20260827T163800Z-1-004
대학부 데이터-20260827T163800Z-1-005
대학부 데이터-20260827T163800Z-1-006
대학부 데이터-20260827T163800Z-1-007
대학부 데이터-20260827T163800Z-1-008
대학부 데이터-20260827T163800Z-1-009
대학부 데이터-20260827T163800Z-1-010
대학부 데이터-20260827T163800Z-1-011
대학부 데이터-20260827T163800Z-1-012


In [ ]:
train_wavs = []
train_labels = []

val_wavs = []
val_labels = []

for part in parts:
    base = part / "대학부 데이터"

    # Training
    train_audio_dir = base / "Training" / "1.원천데이터" / "TS_서울_구급"
    train_label_dir = base / "Training" / "2.라벨링데이터" / "TL_서울_구급"

    # Validation
    val_audio_dir = base / "Validation" / "1.원천데이터" / "VS_서울_구급"
    val_label_dir = base / "Validation" / "2.라벨링데이터" / "VL_서울_구급"

    train_wavs.extend(train_audio_dir.rglob("*.wav"))
    train_labels.extend(train_label_dir.rglob("*.json"))

    val_wavs.extend(val_audio_dir.rglob("*.wav"))
    val_labels.extend(val_label_dir.rglob("*.json"))

print("Train wav :", len(train_wavs))
print("Train json:", len(train_labels))
print("Val wav   :", len(val_wavs))
print("Val json  :", len(val_labels))

Train wav : 29200
Train json: 29200
Val wav   : 3640
Val json  : 3640


In [ ]:
train_wav_stems = {p.stem for p in train_wavs}
train_json_stems = {p.stem for p in train_labels}

val_wav_stems = {p.stem for p in val_wavs}
val_json_stems = {p.stem for p in val_labels}

print("Train wav에만 있음 :", len(train_wav_stems - train_json_stems))
print("Train json에만 있음:", len(train_json_stems - train_wav_stems))

print("Val wav에만 있음   :", len(val_wav_stems - val_json_stems))
print("Val json에만 있음  :", len(val_json_stems - val_wav_stems))

Train wav에만 있음 : 0
Train json에만 있음: 0
Val wav에만 있음   : 0
Val json에만 있음  : 0


## 4. Baseline 1 - Log-Mel Spectrogram + CNN

신고자 발화 waveform에서 Log-Mel spectrogram을 만들고 CNN으로 M/F를 분류했습니다. 긴 통화는 sliding window로 처리하고, Validation에서는 통화별 window 예측을 평균합니다. 아래는 실제 CNN 학습 코드와 epoch별 로그입니다.

In [ ]:
%cd /content/drive/MyDrive/mission1

!python train_cnn.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt" \
    --epochs 10 \
    --batch_size 64 \
    --num_workers 8

/content/drive/MyDrive/mission1
[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training samples  : 29,200
Validation samples: 3,640
Device: cuda
Train 다수 클래스 baseline: F (0.5323)
Epoch 1/10: 100% 457/457 [41:38<00:00,  5.47s/it]
Validation: 100% 3640/3640 [36:57<00:00,  1.64it/s]
epoch=1 train_loss=0.5242 train_acc=0.7888 val_acc=0.8720 val_macro_f1=0.8678
Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt
Epoch 2/10: 100% 457/457 [22:28<00:00,  2.95s/it]
Validation: 100% 3640/3640 [10:54<00:00,  5.56it/s]
epoch=2 train_loss=0.2938 train_acc=0.8834 val_acc=0.9137 val_macro_f1=0.9130
Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt
Epoch 3/10: 100% 457/457 [22:28<00:00,  2.95s/it]
Validation: 100% 3640/3640 [12:05<00:00,  5.0

## 5. Baseline 2 - MFCC + F0 + RandomForest

MFCC, delta, delta-delta의 통계 특징을 사용하고, 성별 분류에서 pitch 정보의 기여를 확인하기 위해 F0 포함/미포함 ablation을 수행했습니다. 기존 결과에서 F0 포함 모델의 Validation Accuracy는 0.9423, F0 미포함 모델은 0.9365였습니다. 이 비교는 F0 특징을 추가한 설정이 이 Validation split에서 더 높은 성능을 보였음을 보여줍니다.

In [ ]:
#F0
%cd /content/drive/MyDrive/mission1

!python train_mfcc_rf.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_mfcc_rf.joblib"

/content/drive/MyDrive/mission1
[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training pairs  : 29,200
Validation pairs: 3,640
Train feature extraction: 100% 29200/29200 [5:42:46<00:00,  1.42it/s]
Val feature extraction: 100% 3640/3640 [38:59<00:00,  1.56it/s]
Train feature shape: (29200, 246)
Val feature shape  : (3640, 246)
F0 사용: True
RandomForest 학습 시작...
[참고] Train accuracy: 1.0000
Validation accuracy : 0.9423
Validation macro F1 : 0.9419

Validation classification report
              precision    recall  f1-score   support

           M     0.9428    0.9316    0.9372      1681
           F     0.9419    0.9515    0.9467      1959

    accuracy                         0.9423      3640
   macro avg     0.9423    0.9415    0.9419      3640
weighted avg     0.9423    0.9423    0.94

In [ ]:
!python train_mfcc_rf.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --no_f0 \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_mfcc_rf_no_f0.joblib"

[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training pairs  : 29,200
Validation pairs: 3,640
Train feature extraction: 100% 29200/29200 [44:09<00:00, 11.02it/s]
Val feature extraction: 100% 3640/3640 [05:29<00:00, 11.05it/s]
Train feature shape: (29200, 240)
Val feature shape  : (3640, 240)
F0 사용: False
RandomForest 학습 시작...
[참고] Train accuracy: 1.0000
Validation accuracy : 0.9365
Validation macro F1 : 0.9361

Validation classification report
              precision    recall  f1-score   support

           M     0.9394    0.9221    0.9307      1681
           F     0.9342    0.9490    0.9415      1959

    accuracy                         0.9365      3640
   macro avg     0.9368    0.9355    0.9361      3640
weighted avg     0.9366    0.9365    0.9365      3640

Train 다수 클래스 baseli

## 6. Proposed Model - WavLM-base-plus

사전학습 encoder로 `microsoft/wavlm-base-plus`를 사용했습니다. 각 음성의 annotation에서 `startAt`, `endAt`, `Speaker` 정보를 이용해 신고자 발화 구간만 추출하고, 해당 waveform을 WavLM encoder에 입력했습니다. 마지막 hidden representation에 mean pooling을 적용하여 768차원 embedding을 생성했습니다.

Training 29,200개와 Validation 3,640개의 embedding을 각각 동일한 방식으로 추출하여 cache에 저장했습니다. 실제 학습 output에서도 Training embedding shape `(29200, 768)`, Validation embedding shape `(3640, 768)`를 확인할 수 있습니다.

신고자 waveform → WavLM-base-plus → 마지막 hidden representation mean pooling → 768-d embedding → StandardScaler → MLP head → M/F

Embedding 추출 후 classification 단계에서는 StandardScaler와 MLP head를 사용했습니다. MLP는 Linear(768 → 256), ReLU, Dropout(0.2), Linear(256 → 2)로 구성했습니다. 아래 Validation embedding 추출 코드는 embedding 생성 과정을 보여주며, 이어지는 head 학습 셀에서는 동일한 방식으로 생성된 Training/Validation embedding cache를 불러와 학습합니다. 실제 head 학습 epoch 로그는 그대로 보존했습니다.

In [ ]:
%cd /content/drive/MyDrive/mission1

import torch
import numpy as np
from pathlib import Path
from transformers import AutoFeatureExtractor, AutoModel

from train_wavlm import collect_split_pairs, extract_embeddings

MODEL_NAME = "microsoft/wavlm-base-plus"

DATA_ROOT = Path("/content/drive/MyDrive/data(inho)")
CACHE_DIR = Path("/content/drive/MyDrive/mission1/wavlm_embedding_cache")

VAL_CACHE = CACHE_DIR / "val_microsoft_wavlm-base-plus_8s_1chunks.npz"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Validation만 탐색
print("\nValidation 데이터 찾는 중...")
val_samples = collect_split_pairs(DATA_ROOT, "Validation")
print(f"Validation samples: {len(val_samples):,}")

# 혹시 이미 validation cache가 있으면 재추출하지 않음
if VAL_CACHE.exists():
    print("\n✅ Validation cache가 이미 존재합니다.")
    print(VAL_CACHE)

else:
    print("\nWavLM 로드 중...")

    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
    encoder.eval()

    for p in encoder.parameters():
        p.requires_grad = False

    print("\n=== Validation WavLM embedding 추출 ===")

    x_val, y_val = extract_embeddings(
        val_samples,
        feature_extractor=feature_extractor,
        encoder=encoder,
        device=device,
        max_seconds=8,
        chunks_per_call=1,
        timestamp_unit="auto",
        desc="Val WavLM embedding",
    )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        VAL_CACHE,
        x=x_val,
        y=y_val,
    )

    print("\n✅ Validation embedding 완료!")
    print("shape:", x_val.shape)
    print("labels:", y_val.shape)
    print("저장 위치:", VAL_CACHE)

/content/drive/MyDrive/mission1
Device: cpu

Validation 데이터 찾는 중...
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Validation samples: 3,640

WavLM 로드 중...


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

... (2 outputs left out)


=== Validation WavLM embedding 추출 ===


Val WavLM embedding:   0%|          | 0/3640 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(



✅ Validation embedding 완료!
shape: (3640, 768)
labels: (3640,)
저장 위치: /content/drive/MyDrive/mission1/wavlm_embedding_cache/val_microsoft_wavlm-base-plus_8s_1chunks.npz


In [ ]:
%cd /content/drive/MyDrive/mission1

import joblib
import numpy as np
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


# =========================
# 설정
# =========================

CACHE_DIR = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache"
)

TRAIN_CACHE = CACHE_DIR / (
    "train_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

VAL_CACHE = CACHE_DIR / (
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

CKPT_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

SCALER_PATH = CKPT_PATH.with_suffix(".scaler.joblib")

EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
HIDDEN_DIM = 256
DROPOUT = 0.2
SEED = 42


# =========================
# seed / device
# =========================

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# =========================
# 캐시 확인
# =========================

assert TRAIN_CACHE.exists(), (
    f"Train cache가 없습니다: {TRAIN_CACHE}"
)

assert VAL_CACHE.exists(), (
    f"Validation cache가 없습니다: {VAL_CACHE}"
)

print("✅ Train cache 확인:", TRAIN_CACHE)
print("✅ Val cache 확인  :", VAL_CACHE)


# =========================
# embedding 로드
# =========================

print("\nEmbedding cache 로드 중...")

train_data = np.load(TRAIN_CACHE)
val_data = np.load(VAL_CACHE)

x_train = train_data["x"].astype(np.float32)
y_train = train_data["y"].astype(np.int64)

x_val = val_data["x"].astype(np.float32)
y_val = val_data["y"].astype(np.int64)

print("Train embedding shape:", x_train.shape)
print("Train label shape    :", y_train.shape)

print("Val embedding shape  :", x_val.shape)
print("Val label shape      :", y_val.shape)


# =========================
# 데이터 sanity check
# =========================

print("\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)

for u, c in zip(unique, counts):
    print(f"class {u}: {c}")

print("\nValidation label distribution:")
unique, counts = np.unique(y_val, return_counts=True)

for u, c in zip(unique, counts):
    print(f"class {u}: {c}")


# =========================
# StandardScaler
# =========================

scaler = StandardScaler()

x_train = scaler.fit_transform(
    x_train
).astype(np.float32)

x_val = scaler.transform(
    x_val
).astype(np.float32)


# =========================
# Dataset / DataLoader
# =========================

train_dataset = TensorDataset(
    torch.from_numpy(x_train),
    torch.from_numpy(y_train),
)

val_dataset = TensorDataset(
    torch.from_numpy(x_val),
    torch.from_numpy(y_val),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# =========================
# Classification Head
# =========================

class Head(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        dropout=0.2,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


model = Head(
    input_dim=x_train.shape[1],
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(device)


# =========================
# 평가 함수
# =========================

def evaluate(model, loader):

    model.eval()

    all_pred = []
    all_true = []

    with torch.inference_mode():

        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            logits = model(x)

            pred = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            all_pred.extend(pred.tolist())
            all_true.extend(y.numpy().tolist())

    acc = accuracy_score(
        all_true,
        all_pred
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro"
    )

    return (
        acc,
        macro_f1,
        np.asarray(all_true),
        np.asarray(all_pred),
    )


# =========================
# 학습 준비
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = nn.CrossEntropyLoss()

best_f1 = -1.0
best_acc = -1.0
best_epoch = -1

CKPT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


# =========================
# 학습
# =========================

print(
    "\n=== Frozen WavLM embedding "
    "+ classification head 학습 ==="
)

for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:

        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(x)

        loss = criterion(
            logits,
            y
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item()
            * y.size(0)
        )

        correct += (
            logits.argmax(dim=1)
            == y
        ).sum().item()

        total += y.size(0)

    train_loss = (
        running_loss
        / total
    )

    train_acc = (
        correct
        / total
    )

    (
        val_acc,
        val_f1,
        _,
        _
    ) = evaluate(
        model,
        val_loader
    )

    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.4f} "
        f"val_acc={val_acc:.4f} "
        f"val_macro_f1={val_f1:.4f}"
    )

    # best macro F1 기준 저장
    if val_f1 > best_f1:

        best_f1 = val_f1
        best_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "model_name":
                    "microsoft/wavlm-base-plus",

                "head_state_dict":
                    model.state_dict(),

                "input_dim":
                    x_train.shape[1],

                "hidden_dim":
                    HIDDEN_DIM,

                "dropout":
                    DROPOUT,

                "max_seconds":
                    8,

                "chunks_per_call":
                    1,

                "best_epoch":
                    best_epoch,

                "best_val_accuracy":
                    best_acc,

                "best_val_macro_f1":
                    best_f1,
            },
            CKPT_PATH,
        )

        joblib.dump(
            scaler,
            SCALER_PATH
        )

        print(
            "  ✅ Best checkpoint 저장:",
            CKPT_PATH
        )


# =========================
# BEST 모델 다시 로드
# =========================

checkpoint = torch.load(
    CKPT_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["head_state_dict"]
)


# =========================
# 최종 Validation 평가
# =========================

(
    final_acc,
    final_f1,
    y_true,
    y_pred,
) = evaluate(
    model,
    val_loader
)

print("\n============================")
print(" BEST VALIDATION RESULT")
print("============================")

print(
    "Best epoch          :",
    checkpoint["best_epoch"]
)

print(
    f"Validation accuracy : "
    f"{final_acc:.4f}"
)

print(
    f"Validation macro F1 : "
    f"{final_f1:.4f}"
)

print("\nClassification Report")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "class 0",
            "class 1"
        ],
        digits=4,
    )
)

print("\n✅ checkpoint:")
print(CKPT_PATH)

print("\n✅ scaler:")
print(SCALER_PATH)

/content/drive/MyDrive/mission1
Device: cpu
✅ Train cache 확인: /content/drive/MyDrive/mission1/wavlm_embedding_cache/train_microsoft_wavlm-base-plus_8s_1chunks.npz
✅ Val cache 확인  : /content/drive/MyDrive/mission1/wavlm_embedding_cache/val_microsoft_wavlm-base-plus_8s_1chunks.npz

Embedding cache 로드 중...
Train embedding shape: (29200, 768)
Train label shape    : (29200,)
Val embedding shape  : (3640, 768)
Val label shape      : (3640,)

Train label distribution:
class 0: 13658
class 1: 15542

Validation label distribution:
class 0: 1681
class 1: 1959

=== Frozen WavLM embedding + classification head 학습 ===
epoch=01 train_loss=0.2070 train_acc=0.9212 val_acc=0.9437 val_macro_f1=0.9433
  ✅ Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_wavlm_1chunk.pt
epoch=02 train_loss=0.1509 train_acc=0.9465 val_acc=0.9495 val_macro_f1=0.9492
  ✅ Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_wavlm_1chunk.pt
epoch=03 train_loss=0.1389 train_acc=0.95

## 7. 3-Chunk Inference

기존 1-chunk 방식은 긴 신고자 음성에서도 일부 구간만 사용한다는 한계가 있습니다. 이를 보완하기 위해 신고자 발화를 시간 순서대로 연결한 waveform이 8초보다 긴 경우 front 8초, middle 8초, end 8초의 세 chunk를 사용했습니다. 각 chunk를 동일한 `microsoft/wavlm-base-plus` encoder에 입력해 embedding을 생성하고, 세 embedding의 평균을 해당 통화의 representation으로 사용했습니다. 8초 이하 waveform은 zero-padding 후 한 번만 처리했습니다.

이 실험에서는 3-chunk용 classification head를 새로 학습하지 않았습니다. 기존 1-chunk 실험에서 학습한 StandardScaler와 MLP classification head를 그대로 사용했으며, Validation inference에서 embedding aggregation 방식만 3-chunk 평균으로 변경했습니다.

기존 notebook의 실제 Validation output에서 WavLM 1-chunk Accuracy는 0.9566, WavLM 3-chunk Accuracy는 0.9750으로 확인되며, Accuracy는 +1.84%p 향상되었습니다.

In [ ]:
%cd /content/drive/MyDrive/mission1

import os
import json
import torch
import numpy as np

from pathlib import Path
from transformers import AutoFeatureExtractor, AutoModel
from tqdm.auto import tqdm

from train_wavlm import collect_split_pairs
from common import SAMPLE_RATE, load_annotation, load_caller_audio


# =========================================================
# 설정
# =========================================================

MODEL_NAME = "microsoft/wavlm-base-plus"

DATA_ROOT = Path(
    "/content/drive/MyDrive/data(inho)"
)

SAVE_DIR = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache"
)

FINAL_CACHE = SAVE_DIR / (
    "val_microsoft_wavlm-base-plus_8s_3chunks.npz"
)

PROGRESS_FILE = SAVE_DIR / (
    "val_microsoft_wavlm-base-plus_8s_3chunks_progress.npz"
)

META_FILE = SAVE_DIR / (
    "val_microsoft_wavlm-base-plus_8s_3chunks_progress.json"
)

MAX_SECONDS = 8
CHUNKS_PER_CALL = 3

SAVE_EVERY = 200

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 3-chunk 함수
# =========================================================

def make_three_chunks(
    waveform,
    max_samples,
):
    waveform = torch.as_tensor(
        waveform,
        dtype=torch.float32
    ).flatten()

    n = waveform.numel()

    # 8초 이하 → padding 후 1번만 처리
    if n <= max_samples:

        if n < max_samples:
            waveform = torch.nn.functional.pad(
                waveform,
                (0, max_samples - n)
            )

        return [waveform]

    # 8초 초과 → 앞 / 가운데 / 뒤 3개
    max_start = n - max_samples

    starts = [
        0,
        max_start // 2,
        max_start
    ]

    return [
        waveform[
            start:start + max_samples
        ]
        for start in starts
    ]


# =========================================================
# Validation pair 수집
# =========================================================

print("\nValidation 데이터 찾는 중...")

val_samples = collect_split_pairs(
    DATA_ROOT,
    "Validation"
)

print(
    f"Validation samples: "
    f"{len(val_samples):,}"
)


# =========================================================
# 이미 최종 cache 있으면 종료
# =========================================================

if FINAL_CACHE.exists():

    print("\n✅ 이미 최종 3-chunk cache가 존재합니다.")
    print(FINAL_CACHE)

else:

    # =====================================================
    # 이어서 실행할 progress 확인
    # =====================================================

    embeddings = []
    labels = []
    processed_stems = []

    start_index = 0

    if (
        PROGRESS_FILE.exists()
        and META_FILE.exists()
    ):

        print(
            "\n✅ 기존 progress 발견."
            " 이어서 실행합니다."
        )

        progress = np.load(
            PROGRESS_FILE
        )

        embeddings = list(
            progress["x"]
        )

        labels = list(
            progress["y"]
        )

        with open(
            META_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            meta = json.load(f)

        processed_stems = meta[
            "processed_stems"
        ]

        start_index = len(
            processed_stems
        )

        print(
            f"이미 완료: "
            f"{start_index:,}"
        )

    else:

        print(
            "\n새 extraction을 시작합니다."
        )


    # =====================================================
    # WavLM 로드
    # =====================================================

    print("\nWavLM 로드 중...")

    feature_extractor = (
        AutoFeatureExtractor
        .from_pretrained(
            MODEL_NAME
        )
    )

    encoder = (
        AutoModel
        .from_pretrained(
            MODEL_NAME
        )
        .to(device)
    )

    encoder.eval()

    for p in encoder.parameters():
        p.requires_grad = False

    print("WavLM 로드 완료.")


    # =====================================================
    # extraction
    # =====================================================

    max_samples = int(
        SAMPLE_RATE
        * MAX_SECONDS
    )

    remaining = val_samples[
        start_index:
    ]

    progress_bar = tqdm(
        remaining,
        initial=start_index,
        total=len(val_samples),
        desc="Val WavLM 3-chunk"
    )

    for idx, sample in enumerate(
        progress_bar,
        start=start_index
    ):

        audio_path = sample[0]
        json_path = sample[1]

        annotation = load_annotation(
            json_path
        )

        gender = str(
            annotation.get(
                "gender",
                ""
            )
        ).strip().upper()

        # 기존 train_wavlm의 label 순서와 동일
        if gender == "M":
            label = 0
        elif gender == "F":
            label = 1
        else:
            raise ValueError(
                f"잘못된 gender: "
                f"{json_path} -> "
                f"{gender}"
            )


        # ---------------------------------------------
        # caller audio
        # ---------------------------------------------

        waveform = load_caller_audio(
            audio_path,
            annotation,
            timestamp_unit="auto"
        )


        # ---------------------------------------------
        # 앞 / 중간 / 뒤 8초
        # ---------------------------------------------

        chunks = make_three_chunks(
            waveform,
            max_samples
        )

        chunk_embeddings = []


        # ---------------------------------------------
        # WavLM 각각 통과
        # ---------------------------------------------

        for chunk in chunks:

            audio_np = (
                chunk
                .detach()
                .cpu()
                .numpy()
            )

            inputs = (
                feature_extractor(
                    audio_np,
                    sampling_rate=SAMPLE_RATE,
                    return_tensors="pt",
                    padding=True
                )
            )

            input_values = (
                inputs[
                    "input_values"
                ]
                .to(device)
            )

            attention_mask = (
                inputs.get(
                    "attention_mask"
                )
            )

            if attention_mask is not None:
                attention_mask = (
                    attention_mask
                    .to(device)
                )


            with torch.inference_mode():

                outputs = encoder(
                    input_values=
                        input_values,
                    attention_mask=
                        attention_mask
                )

                hidden = (
                    outputs
                    .last_hidden_state
                )


                # -------------------------------------
                # masked mean pooling
                # -------------------------------------

                if attention_mask is None:

                    pooled = (
                        hidden
                        .mean(dim=1)
                    )

                else:

                    feature_mask = (
                        encoder
                        ._get_feature_vector_attention_mask(
                            hidden.shape[1],
                            attention_mask
                        )
                    )

                    mask = (
                        feature_mask
                        .unsqueeze(-1)
                        .to(
                            hidden.dtype
                        )
                    )

                    pooled = (
                        (
                            hidden
                            * mask
                        )
                        .sum(dim=1)
                        /
                        mask
                        .sum(dim=1)
                        .clamp(
                            min=1.0
                        )
                    )


            chunk_embeddings.append(
                pooled
                .squeeze(0)
                .cpu()
                .numpy()
            )


        # ---------------------------------------------
        # 세 chunk embedding 평균
        # ---------------------------------------------

        call_embedding = np.mean(
            np.stack(
                chunk_embeddings,
                axis=0
            ),
            axis=0
        ).astype(
            np.float32
        )


        embeddings.append(
            call_embedding
        )

        labels.append(
            label
        )

        processed_stems.append(
            audio_path.stem
        )


        # ---------------------------------------------
        # 중간 저장
        # ---------------------------------------------

        completed = idx + 1

        if (
            completed % SAVE_EVERY == 0
            or completed == len(
                val_samples
            )
        ):

            x_tmp = np.stack(
                embeddings,
                axis=0
            )

            y_tmp = np.asarray(
                labels,
                dtype=np.int64
            )

            np.savez_compressed(
                PROGRESS_FILE,
                x=x_tmp,
                y=y_tmp
            )

            with open(
                META_FILE,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    {
                        "processed_stems":
                            processed_stems
                    },
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            print(
                f"\n💾 Progress 저장: "
                f"{completed:,}/"
                f"{len(val_samples):,}"
            )


    # =====================================================
    # 최종 저장
    # =====================================================

    x_val_3chunk = np.stack(
        embeddings,
        axis=0
    )

    y_val_3chunk = np.asarray(
        labels,
        dtype=np.int64
    )

    np.savez_compressed(
        FINAL_CACHE,
        x=x_val_3chunk,
        y=y_val_3chunk
    )

    print(
        "\n✅ 3-chunk Validation "
        "embedding 완료"
    )

    print(
        "shape:",
        x_val_3chunk.shape
    )

    print(
        "labels:",
        y_val_3chunk.shape
    )

    print(
        "저장 위치:",
        FINAL_CACHE
    )


    # progress 파일 정리
    if PROGRESS_FILE.exists():
        PROGRESS_FILE.unlink()

    if META_FILE.exists():
        META_FILE.unlink()

    print(
        "\n✅ progress 파일 정리 완료"
    )

In [ ]:
%cd /content/drive/MyDrive/mission1

import numpy as np
import joblib
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

from train_wavlm import Head
from common import INDEX_TO_GENDER


# =========================================================
# 경로
# =========================================================

CACHE_1 = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

CACHE_3 = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_3chunks.npz"
)

CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

SCALER_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)


# =========================================================
# 데이터 로드
# =========================================================

data1 = np.load(CACHE_1)
data3 = np.load(CACHE_3)

x1 = data1["x"].astype(np.float32)
y1 = data1["y"].astype(np.int64)

x3 = data3["x"].astype(np.float32)
y3 = data3["y"].astype(np.int64)

print("1-chunk:", x1.shape)
print("3-chunk:", x3.shape)

assert np.array_equal(y1, y3)

print("✅ label 순서 일치")


# =========================================================
# scaler
# =========================================================

scaler = joblib.load(SCALER_PATH)

x1_scaled = scaler.transform(x1).astype(np.float32)
x3_scaled = scaler.transform(x3).astype(np.float32)


# =========================================================
# WavLM head 로드
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CKPT,
    map_location=device,
    weights_only=False
)

model = Head(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"]
).to(device)

model.load_state_dict(
    checkpoint["head_state_dict"]
)

model.eval()


# =========================================================
# probability 함수
# =========================================================

def predict_prob(x):

    probs = []

    batch_size = 512

    with torch.inference_mode():

        for start in range(
            0,
            len(x),
            batch_size
        ):

            batch = torch.from_numpy(
                x[start:start + batch_size]
            ).to(device)

            logits = model(batch)

            prob = torch.softmax(
                logits,
                dim=1
            )

            probs.append(
                prob.cpu().numpy()
            )

    return np.vstack(probs)


# =========================================================
# 예측
# =========================================================

prob1 = predict_prob(x1_scaled)
prob3 = predict_prob(x3_scaled)

pred1 = prob1.argmax(axis=1)
pred3 = prob3.argmax(axis=1)


# =========================================================
# 성능
# =========================================================

acc1 = accuracy_score(y1, pred1)
f1_1 = f1_score(
    y1,
    pred1,
    average="macro"
)

acc3 = accuracy_score(y3, pred3)
f1_3 = f1_score(
    y3,
    pred3,
    average="macro"
)


print("\n============================")
print(" WavLM 비교")
print("============================")

print(
    f"1-chunk | "
    f"acc={acc1:.4f} | "
    f"macro_f1={f1_1:.4f}"
)

print(
    f"3-chunk | "
    f"acc={acc3:.4f} | "
    f"macro_f1={f1_3:.4f}"
)

print("\nAccuracy 변화:")
print(
    f"{acc1:.4f} → {acc3:.4f} "
    f"({(acc3-acc1)*100:+.2f}%p)"
)


print("\n=== 3-chunk Classification Report ===")

target_names = [
    INDEX_TO_GENDER[i]
    for i in range(len(INDEX_TO_GENDER))
]

print(
    classification_report(
        y3,
        pred3,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

/content/drive/MyDrive/mission1
1-chunk: (3640, 768)
3-chunk: (3640, 768)
✅ label 순서 일치

 WavLM 비교
1-chunk | acc=0.9566 | macro_f1=0.9563
3-chunk | acc=0.9750 | macro_f1=0.9748

Accuracy 변화:
0.9566 → 0.9750 (+1.84%p)

=== 3-chunk Classification Report ===
              precision    recall  f1-score   support

           M     0.9783    0.9673    0.9728      1681
           F     0.9722    0.9816    0.9769      1959

    accuracy                         0.9750      3640
   macro avg     0.9753    0.9745    0.9748      3640
weighted avg     0.9750    0.9750    0.9750      3640



## 8. Late Fusion 실험

Late fusion은 WavLM과 MFCC+F0 RandomForest가 출력한 class probability의 가중 평균으로 최종 예측을 만드는 방식입니다.

### 1-chunk fusion

먼저 WavLM 1-chunk와 MFCC+F0 RandomForest의 class probability를 가중 평균하는 late fusion을 실험했습니다. RandomForest Accuracy는 0.9423, WavLM 1-chunk Accuracy는 0.9566이었으며, WavLM weight 0.45와 RF weight 0.55에서 Accuracy 0.9635, Macro F1 0.9632를 기록했습니다. 이 결과는 두 모델의 probability를 결합했을 때 Validation 성능이 향상되는 조합이 존재함을 보여주지만, 일반화 성능의 향상을 단정하는 결과는 아닙니다.

### 3-chunk fusion

이후 성능이 향상된 WavLM 3-chunk에도 동일한 late fusion 방식을 적용했습니다. MFCC+F0 RandomForest는 Accuracy 0.9423, Macro F1 0.9419를 기록했고, WavLM 3-chunk 단독은 Accuracy 0.9750, Macro F1 0.9748을 기록했습니다. WavLM weight 0.85와 RF weight 0.15를 적용한 fusion의 Accuracy는 0.9755, Macro F1은 0.9754였습니다.

Fusion 적용으로 WavLM 3-chunk 단독 예측 중 총 12개가 변경되었습니다. 이 중 오답에서 정답으로 바뀐 경우는 7개, 정답에서 오답으로 바뀐 경우는 5개로 순개선은 2개였습니다. WavLM 3-chunk 단독 대비 Accuracy 차이는 0.0005, 즉 0.05%p입니다.

Fusion weight는 동일한 Validation set의 성능을 기준으로 탐색·선택했습니다. 따라서 fusion의 0.9755는 별도의 독립 test set에서 검증된 일반화 성능으로 해석하지 않습니다.

In [ ]:
%cd /content/drive/MyDrive/mission1

import joblib
import numpy as np
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

from train_mfcc_rf import collect_split_pairs, build_features
from train_wavlm import Head
from common import INDEX_TO_GENDER


# =========================================================
# 경로 설정
# =========================================================

DATA_ROOT = Path("/content/drive/MyDrive/data(inho)")

RF_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_mfcc_rf.joblib"
)

WAVLM_VAL_CACHE = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

WAVLM_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

WAVLM_SCALER = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)


# =========================================================
# 1. RandomForest 모델 로드
# =========================================================

rf_data = joblib.load(RF_CKPT)

rf_model = rf_data["classifier"]
n_mfcc = rf_data["n_mfcc"]
use_f0 = rf_data["use_f0"]
timestamp_unit = rf_data["timestamp_unit"]

print("=== RF 모델 설정 ===")
print("n_mfcc        :", n_mfcc)
print("use_f0        :", use_f0)
print("timestamp_unit:", timestamp_unit)
print("classes       :", rf_model.classes_)


# =========================================================
# 2. Validation MFCC + F0 feature 추출
# =========================================================

print("\n=== Validation MFCC+F0 feature 추출 ===")

val_pairs = collect_split_pairs(
    DATA_ROOT,
    "Validation"
)

x_val_rf, y_val_rf = build_features(
    val_pairs,
    n_mfcc=n_mfcc,
    use_f0=use_f0,
    timestamp_unit=timestamp_unit,
    desc="Validation RF feature",
)

print("\nRF feature shape:", x_val_rf.shape)
print("RF label shape  :", y_val_rf.shape)


# =========================================================
# 3. RF 확률 예측
# =========================================================

rf_prob = rf_model.predict_proba(x_val_rf)

print("RF probability shape:", rf_prob.shape)


# =========================================================
# 4. WavLM Validation embedding 로드
# =========================================================

wavlm_data = np.load(WAVLM_VAL_CACHE)

x_val_wavlm = wavlm_data["x"].astype(np.float32)
y_val_wavlm = wavlm_data["y"].astype(np.int64)

print("\nWavLM embedding shape:", x_val_wavlm.shape)
print("WavLM label shape    :", y_val_wavlm.shape)


# =========================================================
# 5. Validation 순서 일치 확인
# =========================================================

assert len(y_val_rf) == len(y_val_wavlm), (
    "RF와 WavLM Validation 샘플 수가 다릅니다."
)

assert np.array_equal(y_val_rf, y_val_wavlm), (
    "RF와 WavLM Validation 순서가 다릅니다."
)

print("\n✅ RF / WavLM Validation 순서 일치")


# =========================================================
# 6. WavLM scaler 적용
# =========================================================

scaler = joblib.load(WAVLM_SCALER)

x_val_scaled = scaler.transform(
    x_val_wavlm
).astype(np.float32)


# =========================================================
# 7. WavLM classification head 로드
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

checkpoint = torch.load(
    WAVLM_CKPT,
    map_location=device,
    weights_only=False,
)

wavlm_model = Head(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"],
).to(device)

wavlm_model.load_state_dict(
    checkpoint["head_state_dict"]
)

wavlm_model.eval()


# =========================================================
# 8. WavLM 확률 예측
# =========================================================

wavlm_probs = []

batch_size = 512

with torch.inference_mode():

    for start in range(
        0,
        len(x_val_scaled),
        batch_size
    ):

        x = torch.from_numpy(
            x_val_scaled[start:start + batch_size]
        ).to(device)

        logits = wavlm_model(x)

        prob = torch.softmax(
            logits,
            dim=1
        )

        wavlm_probs.append(
            prob.cpu().numpy()
        )

wavlm_prob = np.vstack(wavlm_probs)

print("WavLM probability shape:", wavlm_prob.shape)


# =========================================================
# 9. 단독 모델 성능 확인
# =========================================================

y_true = y_val_wavlm

rf_pred = rf_prob.argmax(axis=1)
wavlm_pred = wavlm_prob.argmax(axis=1)

rf_acc = accuracy_score(
    y_true,
    rf_pred
)

rf_f1 = f1_score(
    y_true,
    rf_pred,
    average="macro"
)

wavlm_acc = accuracy_score(
    y_true,
    wavlm_pred
)

wavlm_f1 = f1_score(
    y_true,
    wavlm_pred,
    average="macro"
)

print("\n============================")
print(" 단독 모델 성능")
print("============================")

print(
    f"RF     | acc={rf_acc:.4f} "
    f"| macro_f1={rf_f1:.4f}"
)

print(
    f"WavLM  | acc={wavlm_acc:.4f} "
    f"| macro_f1={wavlm_f1:.4f}"
)


# =========================================================
# 10. Late Fusion
# =========================================================

print("\n============================")
print(" Late Fusion")
print("============================")

results = []

for wavlm_weight in np.arange(
    0.0,
    1.01,
    0.05
):

    rf_weight = 1.0 - wavlm_weight

    fusion_prob = (
        wavlm_weight * wavlm_prob
        + rf_weight * rf_prob
    )

    fusion_pred = fusion_prob.argmax(
        axis=1
    )

    acc = accuracy_score(
        y_true,
        fusion_pred
    )

    macro_f1 = f1_score(
        y_true,
        fusion_pred,
        average="macro"
    )

    results.append(
        (
            wavlm_weight,
            rf_weight,
            acc,
            macro_f1,
        )
    )

    print(
        f"WavLM={wavlm_weight:.2f} "
        f"RF={rf_weight:.2f} "
        f"| acc={acc:.4f} "
        f"| macro_f1={macro_f1:.4f}"
    )


# =========================================================
# 11. 최고 Fusion 조합 찾기
# =========================================================

best = max(
    results,
    key=lambda x: x[3]
)

best_wavlm_weight = best[0]
best_rf_weight = best[1]
best_acc = best[2]
best_f1 = best[3]

print("\n============================")
print(" BEST FUSION")
print("============================")

print(
    f"WavLM weight : {best_wavlm_weight:.2f}"
)

print(
    f"RF weight    : {best_rf_weight:.2f}"
)

print(
    f"Accuracy     : {best_acc:.4f}"
)

print(
    f"Macro F1     : {best_f1:.4f}"
)


# =========================================================
# 12. 최고 Fusion classification report
# =========================================================

best_prob = (
    best_wavlm_weight * wavlm_prob
    + best_rf_weight * rf_prob
)

best_pred = best_prob.argmax(
    axis=1
)

target_names = [
    INDEX_TO_GENDER[i]
    for i in range(len(INDEX_TO_GENDER))
]

print("\n=== BEST FUSION Classification Report ===")

print(
    classification_report(
        y_true,
        best_pred,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )
)

/content/drive/MyDrive/mission1
=== RF 모델 설정 ===
n_mfcc        : 20
use_f0        : True
timestamp_unit: auto
classes       : [0 1]

=== Validation MFCC+F0 feature 추출 ===
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0


Validation RF feature: 100%|██████████| 3640/3640 [14:11<00:00,  4.28it/s]



RF feature shape: (3640, 246)
RF label shape  : (3640,)
RF probability shape: (3640, 2)

WavLM embedding shape: (3640, 768)
WavLM label shape    : (3640,)

✅ RF / WavLM Validation 순서 일치
Device: cpu
WavLM probability shape: (3640, 2)

 단독 모델 성능
RF     | acc=0.9423 | macro_f1=0.9419
WavLM  | acc=0.9566 | macro_f1=0.9563

 Late Fusion
WavLM=0.00 RF=1.00 | acc=0.9423 | macro_f1=0.9419
WavLM=0.05 RF=0.95 | acc=0.9451 | macro_f1=0.9447
WavLM=0.10 RF=0.90 | acc=0.9478 | macro_f1=0.9475
WavLM=0.15 RF=0.85 | acc=0.9530 | macro_f1=0.9527
WavLM=0.20 RF=0.80 | acc=0.9547 | macro_f1=0.9544
WavLM=0.25 RF=0.75 | acc=0.9580 | macro_f1=0.9577
WavLM=0.30 RF=0.70 | acc=0.9596 | macro_f1=0.9593
WavLM=0.35 RF=0.65 | acc=0.9607 | macro_f1=0.9604
WavLM=0.40 RF=0.60 | acc=0.9624 | macro_f1=0.9621
WavLM=0.45 RF=0.55 | acc=0.9635 | macro_f1=0.9632
WavLM=0.50 RF=0.50 | acc=0.9624 | macro_f1=0.9621
WavLM=0.55 RF=0.45 | acc=0.9629 | macro_f1=0.9626
WavLM=0.60 RF=0.40 | acc=0.9629 | macro_f1=0.9626
WavLM=0.65 RF=0

In [ ]:
%cd /content/drive/MyDrive/mission1

import numpy as np
import joblib
import torch

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

from train_mfcc_rf import collect_split_pairs, build_features
from train_wavlm import Head
from common import INDEX_TO_GENDER


# =========================================================
# 경로
# =========================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/data(inho)"
)

RF_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_mfcc_rf.joblib"
)

WAVLM_CACHE_3 = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_3chunks.npz"
)

WAVLM_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

WAVLM_SCALER = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)


# =========================================================
# 1. RF 로드
# =========================================================

rf_data = joblib.load(RF_CKPT)

rf_model = rf_data["classifier"]

print("=== RF ===")
print("use_f0:", rf_data["use_f0"])


# =========================================================
# 2. Validation RF feature 추출
# =========================================================
# 이전 runtime이 끊겼으므로 RF feature를 다시 만들어야 함.
# WavLM embedding은 다시 추출하지 않음.
# =========================================================

print("\nValidation RF feature 추출...")

val_pairs = collect_split_pairs(
    DATA_ROOT,
    "Validation"
)

x_val_rf, y_val_rf = build_features(
    val_pairs,
    n_mfcc=rf_data["n_mfcc"],
    use_f0=rf_data["use_f0"],
    timestamp_unit=rf_data["timestamp_unit"],
    desc="Validation RF feature",
)

rf_prob = rf_model.predict_proba(
    x_val_rf
)


# =========================================================
# 3. 3-chunk WavLM embedding 로드
# =========================================================

data3 = np.load(WAVLM_CACHE_3)

x3 = data3["x"].astype(
    np.float32
)

y3 = data3["y"].astype(
    np.int64
)

assert np.array_equal(
    y_val_rf,
    y3
)

print(
    "\n✅ RF / WavLM sample 순서 일치"
)


# =========================================================
# 4. WavLM scaler
# =========================================================

scaler = joblib.load(
    WAVLM_SCALER
)

x3_scaled = scaler.transform(
    x3
).astype(np.float32)


# =========================================================
# 5. WavLM head
# =========================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    WAVLM_CKPT,
    map_location=device,
    weights_only=False
)

model = Head(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"]
).to(device)

model.load_state_dict(
    checkpoint["head_state_dict"]
)

model.eval()


# =========================================================
# 6. 3-chunk WavLM probability
# =========================================================

wavlm_probs = []

batch_size = 512

with torch.inference_mode():

    for start in range(
        0,
        len(x3_scaled),
        batch_size
    ):

        batch = torch.from_numpy(
            x3_scaled[
                start:start + batch_size
            ]
        ).to(device)

        logits = model(batch)

        prob = torch.softmax(
            logits,
            dim=1
        )

        wavlm_probs.append(
            prob.cpu().numpy()
        )

wavlm_prob = np.vstack(
    wavlm_probs
)


# =========================================================
# 7. 단독 성능 확인
# =========================================================

y_true = y3

rf_pred = rf_prob.argmax(axis=1)
wavlm_pred = wavlm_prob.argmax(axis=1)

rf_acc = accuracy_score(
    y_true,
    rf_pred
)

wavlm_acc = accuracy_score(
    y_true,
    wavlm_pred
)

print("\n============================")
print(" 단독 모델")
print("============================")

print(
    f"RF             : "
    f"{rf_acc:.4f}"
)

print(
    f"WavLM 3-chunk : "
    f"{wavlm_acc:.4f}"
)


# =========================================================
# 8. Fusion 탐색
# =========================================================
# WavLM weight를 0.00 ~ 1.00까지
# 0.01 간격으로 탐색
# =========================================================

print("\n============================")
print(" 3-chunk + RF Fusion")
print("============================")

results = []

for wavlm_weight in np.arange(
    0.00,
    1.001,
    0.01
):

    rf_weight = (
        1.0 - wavlm_weight
    )

    fusion_prob = (
        wavlm_weight * wavlm_prob
        +
        rf_weight * rf_prob
    )

    fusion_pred = (
        fusion_prob.argmax(
            axis=1
        )
    )

    acc = accuracy_score(
        y_true,
        fusion_pred
    )

    f1 = f1_score(
        y_true,
        fusion_pred,
        average="macro"
    )

    results.append(
        (
            wavlm_weight,
            rf_weight,
            acc,
            f1
        )
    )


# =========================================================
# 9. Accuracy 기준 정렬
# =========================================================

results_sorted = sorted(
    results,
    key=lambda x: (
        x[2],   # Accuracy
        x[3],   # 동률이면 Macro F1
    ),
    reverse=True
)


print("\n=== TOP 10 ===")

for result in results_sorted[:10]:

    w, r, acc, f1 = result

    print(
        f"WavLM={w:.2f} "
        f"RF={r:.2f} "
        f"| acc={acc:.4f} "
        f"| macro_f1={f1:.4f}"
    )


# =========================================================
# 10. BEST
# =========================================================

best = results_sorted[0]

best_wavlm = best[0]
best_rf = best[1]
best_acc = best[2]
best_f1 = best[3]

print("\n============================")
print(" 🏆 BEST FUSION")
print("============================")

print(
    f"WavLM weight : "
    f"{best_wavlm:.2f}"
)

print(
    f"RF weight    : "
    f"{best_rf:.2f}"
)

print(
    f"Accuracy     : "
    f"{best_acc:.4f}"
)

print(
    f"Macro F1     : "
    f"{best_f1:.4f}"
)


# =========================================================
# 11. Classification Report
# =========================================================

best_prob = (
    best_wavlm * wavlm_prob
    +
    best_rf * rf_prob
)

best_pred = best_prob.argmax(
    axis=1
)

target_names = [
    INDEX_TO_GENDER[i]
    for i in range(
        len(INDEX_TO_GENDER)
    )
]

print(
    "\n=== BEST Classification Report ==="
)

print(
    classification_report(
        y_true,
        best_pred,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

## 9. 최종 실험 결과

| Model | Validation Accuracy |
|---|---:|
| Log-Mel + CNN | 0.9396 |
| MFCC without F0 + RandomForest | 0.9365 |
| MFCC + F0 + RandomForest | 0.9423 |
| WavLM 1-Chunk | 0.9566 |
| WavLM 1-Chunk + RF Fusion | 0.9635 |
| WavLM 3-Chunk | 0.9750 |
| WavLM 3-Chunk + RF Fusion | 0.9755 |

최종 모델은 **WavLM-base-plus + 3-chunk inference**입니다. 기존 Validation 결과에서 1-chunk Accuracy 0.9566이 3-chunk inference로 0.9750까지 향상됐습니다. 3-chunk + RF fusion은 작은 차이를 보였지만, validation weight 탐색 결과이므로 최종 제출에는 포함하지 않았습니다.